# 01 · Ingesta desde raw

In [1]:
from concurrent.futures import ThreadPoolExecutor
import pandas as pd

RAIZ = "https://raw.githubusercontent.com/CSSEGISandData/COVID-19/master/csse_covid_19_data/"
BASE = RAIZ + "csse_covid_19_daily_reports/"
BASE_SERIES = RAIZ + "csse_covid_19_time_series/"


fechas = pd.date_range("2020-01-22", "2023-03-09")

## 1 · Lectura
Se decidió usar hilos ya que bajaba el tiempo de 6 min a aproximadamente 20s.

In [2]:
def leer_reporte(f):
    nombre = f.strftime("%m-%d-%Y") + ".csv"  
    return pd.read_csv(BASE + nombre)


with ThreadPoolExecutor(max_workers=10) as ex:
    reportes = dict(zip(fechas, ex.map(leer_reporte, fechas)))  

series = {
    "confirmados_global": pd.read_csv(BASE_SERIES + "time_series_covid19_confirmed_global.csv"),
    "fallecidos_global": pd.read_csv(BASE_SERIES + "time_series_covid19_deaths_global.csv"),
    "recuperados_global": pd.read_csv(BASE_SERIES + "time_series_covid19_recovered_global.csv"),
    "confirmados_us": pd.read_csv(BASE_SERIES + "time_series_covid19_confirmed_US.csv"),
    "fallecidos_us": pd.read_csv(BASE_SERIES + "time_series_covid19_deaths_US.csv"),
}
uid = pd.read_csv(RAIZ + "UID_ISO_FIPS_LookUp_Table.csv")   # llave de territorio, ISO3 y población

In [3]:
# Alternativa sin hilos ~6 min:
# reportes = {}
# for f in fechas:
#     reportes[f] = leer_reporte(f)

### Control: se leyeron las 1.143 fechas

In [4]:
print("Fechas esperadas:", len(fechas), "| leídas:", len(reportes))
print("Filas en total:", sum(len(t) for t in reportes.values()))
for nombre, t in series.items():
    print(f"Serie {nombre}: {len(t)} filas")
print("Tabla UID:", len(uid), "filas")

Fechas esperadas: 1143 | leídas: 1143
Filas en total: 4287473
Serie confirmados_global: 289 filas
Serie fallecidos_global: 289 filas
Serie recuperados_global: 274 filas
Serie confirmados_us: 3342 filas
Serie fallecidos_us: 3342 filas
Tabla UID: 4321 filas


### Cambios de esquema
Agrupamos los reportes por su lista de columnas: cada grupo es un esquema.

In [5]:
columnas = pd.Series({f: ", ".join(t.columns) for f, t in reportes.items()})
columnas.groupby(columnas, sort=False).apply(lambda s: f"{s.index.min().date()} → {s.index.max().date()} ({len(s)} reportes)")

Province/State, Country/Region, Last Update, Confirmed, Deaths, Recovered                                                                                          2020-01-22 → 2020-02-29 (39 reportes)
Province/State, Country/Region, Last Update, Confirmed, Deaths, Recovered, Latitude, Longitude                                                                     2020-03-01 → 2020-03-21 (21 reportes)
FIPS, Admin2, Province_State, Country_Region, Last_Update, Lat, Long_, Confirmed, Deaths, Recovered, Active, Combined_Key                                          2020-03-22 → 2020-05-28 (68 reportes)
FIPS, Admin2, Province_State, Country_Region, Last_Update, Lat, Long_, Confirmed, Deaths, Recovered, Active, Combined_Key, Incidence_Rate, Case-Fatality_Ratio    2020-05-29 → 2020-11-08 (164 reportes)
FIPS, Admin2, Province_State, Country_Region, Last_Update, Lat, Long_, Confirmed, Deaths, Recovered, Active, Combined_Key, Incident_Rate, Case_Fatality_Ratio     2020-11-09 → 2023-03-09 (851 repor

## 2 · Unificación de los 5 esquemas
El diccionario `RENOMBRAR`  traduce cada nombre antiguo a uno solo en snake_case. Además se agregan:
- `fecha_reporte`: la fecha del nombre del archivo ;
- `esquema`: 1 a 5, para saber de qué etapa viene cada fila;
- `archivo_origen` y `fila_origen`: de qué CSV y de qué fila salió cada registro.
  `fila_origen` es la posición en `pd.read_csv` (0 = primera fila de datos), así se comprueba con
  `pd.read_csv(BASE + archivo_origen).iloc[fila_origen]`.

In [6]:
import numpy as np

RENOMBRAR = {
    "Province/State": "province_state", "Province_State": "province_state",
    "Country/Region": "country_region", "Country_Region": "country_region",
    "Last Update": "last_update", "Last_Update": "last_update",
    "Latitude": "lat", "Lat": "lat", "Longitude": "long", "Long_": "long",
    "Incidence_Rate": "incident_rate", "Incident_Rate": "incident_rate",
    "Case-Fatality_Ratio": "case_fatality_ratio",
    "Case_Fatality_Ratio": "case_fatality_ratio",
}

tablas = []
for f, t in reportes.items():
    t = t.rename(columns=RENOMBRAR)
    t.columns = t.columns.str.lower()
    t["fecha_reporte"] = f                                         
    t["archivo_origen"] = f.strftime("%m-%d-%Y") + ".csv"       
    t["fila_origen"] = t.index                                   
    tablas.append(t)

df = pd.concat(tablas, ignore_index=True)
df["last_update"] = pd.to_datetime(df["last_update"], format="mixed").dt.normalize()   

condiciones = [
    df["fecha_reporte"] < "2020-03-01",   
    df["fecha_reporte"] < "2020-03-22",   
    df["fecha_reporte"] < "2020-05-29",   
    df["fecha_reporte"] < "2020-11-09",   
]
df["esquema"] = np.select(condiciones, [1, 2, 3, 4], default=5)   

print("Filas:", len(df), "| columnas:", df.shape[1])
df.dtypes

Filas: 4287473 | columnas: 18


province_state                 object
country_region                 object
last_update            datetime64[ns]
confirmed                     float64
deaths                        float64
recovered                     float64
fecha_reporte          datetime64[ns]
archivo_origen                 object
fila_origen                     int64
lat                           float64
long                          float64
fips                          float64
admin2                         object
active                        float64
combined_key                   object
incident_rate                 float64
case_fatality_ratio           float64
esquema                         int64
dtype: object

### Filas por esquema

In [7]:
df.groupby("esquema").agg(desde=("fecha_reporte", "min"), hasta=("fecha_reporte", "max"),
                          reportes=("fecha_reporte", "nunique"), filas=("fecha_reporte", "size"))

,desde,hasta,reportes,filas
esquema,,,,
1,2020-01-22,2020-02-29,39,4242
2,2020-03-01,2020-03-21,21,5555
3,2020-03-22,2020-05-28,68,217486
4,2020-05-29,2020-11-08,164,643463
5,2020-11-09,2023-03-09,851,3416727


## 3 · Llave de territorio
JHU escribió algunos países de varias formas (`Mainland China` → `China`, `South Korea` → `Korea, South`), y algunos
territorios partieron como país y después pasaron a ser provincia de otro (`Hong Kong` → provincia de `China`).
Los diccionarios `PAIS` y `TERRITORIO` (los mismos de `00_exploracion`) unifican esos nombres.

In [8]:
PAIS = {
    " Azerbaijan": "Azerbaijan", "Bahamas, The": "Bahamas", "The Bahamas": "Bahamas",
    "Cape Verde": "Cabo Verde", "Czech Republic": "Czechia", "East Timor": "Timor-Leste",
    "Gambia, The": "Gambia", "The Gambia": "Gambia", "Iran (Islamic Republic of)": "Iran",
    "Ivory Coast": "Cote d'Ivoire", "Mainland China": "China", "Republic of Ireland": "Ireland",
    "Republic of Korea": "Korea, South", "South Korea": "Korea, South",
    "Republic of Moldova": "Moldova", "Republic of the Congo": "Congo (Brazzaville)",
    "Russian Federation": "Russia", "Taiwan": "Taiwan*", "Taipei and environs": "Taiwan*",
    "Vatican City": "Holy See", "Viet Nam": "Vietnam",
    "Palestine": "West Bank and Gaza", "occupied Palestinian territory": "West Bank and Gaza",
    "North Ireland": "United Kingdom", "Others": "Diamond Princess", "Cruise Ship": "Diamond Princess",
}

# Territorios que primero venían como "país" y después como provincia de otro país.
TERRITORIO = {
    "Hong Kong": "China", "Hong Kong SAR": "China", "Macau": "China", "Macao SAR": "China",
    "Puerto Rico": "US", "Guam": "US",
    "Aruba": "Netherlands", "Curacao": "Netherlands",
    "Cayman Islands": "United Kingdom", "Channel Islands": "United Kingdom",
    "Gibraltar": "United Kingdom", "Jersey": "United Kingdom", "Guernsey": "United Kingdom",
    "Faroe Islands": "Denmark", "Greenland": "Denmark",
    "French Guiana": "France", "Guadeloupe": "France", "Martinique": "France", "Mayotte": "France",
    "Reunion": "France", "Saint Barthelemy": "France", "Saint Martin": "France", "St. Martin": "France",
}

for col in ["country_region", "province_state", "admin2"]:
    df[col] = df[col].str.strip()                       # quita espacios sobrantes 

es_territorio = df["country_region"].isin(TERRITORIO)
df.loc[es_territorio, "province_state"] = df.loc[es_territorio, "country_region"]
df.loc[es_territorio, "country_region"] = df.loc[es_territorio, "country_region"].map(TERRITORIO)
df["country_region"] = df["country_region"].replace(PAIS)

### Cruce con la tabla UID
Cruzamos por **país + provincia + condado** y agregamos la columna `uid`: es la llave para traer
población, ISO3 y coordenadas en el DW. Provincia o condado vacío = `""` (el territorio es el país o la provincia completa).

In [9]:
llave = ["country_region", "province_state", "admin2"]
tabla_uid = uid.rename(columns={"Country_Region": "country_region", "Province_State": "province_state",
                                "Admin2": "admin2", "UID": "uid"})[llave + ["uid"]]
tabla_uid = tabla_uid.fillna({"province_state": "", "admin2": ""})

filas_antes = len(df)
df = df.fillna({"province_state": "", "admin2": ""}).merge(tabla_uid, on=llave, how="left")
print("Filas antes:", filas_antes, "| después:", len(df))   # el cruce no debe duplicar filas

Filas antes: 4287473 | después: 4287473


### Control: cuántas filas cruzan con la tabla UID

In [10]:
cruza = df["uid"].notna()
print(f"Filas que cruzan: {cruza.mean():.2%} ({cruza.sum()} de {len(df)})")
print("Países que no existen en la tabla UID:", sorted(set(df["country_region"]) - set(uid["Country_Region"])) or "ninguno")

Filas que cruzan: 99.93% (4284314 de 4287473)


Países que no existen en la tabla UID:

 ninguno


### Territorios que no cruzan
No se corrigen aquí: se deciden en `02_calidad.ipynb`, sección 4.2 (cruceros y repatriados se excluyen; el resto se marca con `uid` vacío).

In [11]:
no_cruzan = (
    df.loc[~cruza].groupby(llave)
      .agg(filas=("fecha_reporte", "size"), desde=("fecha_reporte", "min"), hasta=("fecha_reporte", "max"))
      .sort_values("filas", ascending=False)
)
print("Territorios que no cruzan:", len(no_cruzan))
no_cruzan.head(30)

Territorios que no cruzan: 183


filas  \
country_region   province_state                              admin2                 
US               Alaska                                      Yakutat          968   
Canada           Repatriated Travellers                                       836   
US               New York                                    New York City    162   
Taiwan*          Taiwan                                                        49   
US               Los Angeles, CA                                               38   
                 San Benito, CA                                                36   
                 Santa Clara, CA                                               35   
                 Tempe, AZ                                                     35   
Canada           Toronto, ON                                                   34   
US               Madison, WI                                                   34   
                 Boston, MA                                                    34   
Canada           London, ON                                                    34   
US               Orange, CA                                                    32   
Diamond Princess Diamond Princess cruise ship                                  31   
US               Chicago, IL                                                   30   
                 Seattle, WA                                                   30   
Australia        From Diamond Princess                                         30   
US               San Diego County, CA                                          28   
                 San Antonio, TX                                               26   
                 Sacramento County, CA                                         18   
                 Humboldt County, CA                                           18   
                 Lackland, TX (From Diamond Princess)                          17   
                 Travis, CA (From Diamond Princess)                            17   
                 Omaha, NE (From Diamond Princess)                             17   
                 Unassigned Location (From Diamond Princess)                   15   
Diamond Princess Diamond Princess                                              13   
Canada           Montreal, QC                                                  13   
Denmark          Denmark                                                       11   
France           France                                                        11   
US               Snohomish County, WA                                          10   

                                                                                desde  \
country_region   province_state                              admin2                     
US               Alaska                                      Yakutat       2020-03-22   
Canada           Repatriated Travellers                                    2020-11-24   
US               New York                                    New York City 2020-03-22   
Taiwan*          Taiwan                                                    2020-01-22   
US               Los Angeles, CA                                           2020-02-01   
                 San Benito, CA                                            2020-02-03   
                 Santa Clara, CA                                           2020-02-01   
                 Tempe, AZ                                                 2020-02-01   
Canada           Toronto, ON                                               2020-02-04   
US               Madison, WI                                               2020-02-05   
                 Boston, MA                                                2020-02-01   
Canada           London, ON                                                2020-02-04   
US               Orange, CA                                                2020-02-01   
Diamond Princess Dia

## 4 · Capa limpia en Parquet
La tabla unificada se guarda en `data/clean/`. 
Se guarda con `compression="zstd"`: con la compresión por defecto de pandas (`snappy`) pesa ~99 MB, justo en el
límite de 100 MB por archivo de GitHub; con Zstandard baja a ~66 MB.

In [12]:
import os

os.makedirs("../data/clean", exist_ok=True)
RUTA_PARQUET = "../data/clean/covid_limpio.parquet"

df.to_parquet(RUTA_PARQUET, index=False, compression="zstd") 

print(f"Parquet guardado: {os.path.getsize(RUTA_PARQUET) / 2**20:.1f} MB")

Parquet guardado: 67.1 MB


### CSV frente a Parquet: tamaño en disco y tiempo de lectura
Guardamos la misma tabla como CSV temporal, medimos los dos archivos y después borramos el CSV
(no se sube al repo).

In [13]:
import time

RUTA_CSV = "../data/clean/covid_limpio_temporal.csv"   # solo para comparar
df.to_csv(RUTA_CSV, index=False)

inicio = time.time()
pd.read_csv(RUTA_CSV)
tiempo_csv = time.time() - inicio

inicio = time.time()
pd.read_parquet(RUTA_PARQUET)
tiempo_parquet = time.time() - inicio

comparacion = pd.DataFrame({
    "formato": ["CSV", "Parquet"],
    "tamaño_MB": [os.path.getsize(RUTA_CSV) / 2**20, os.path.getsize(RUTA_PARQUET) / 2**20],
    "lectura_s": [tiempo_csv, tiempo_parquet],
}).round(1)

os.remove(RUTA_CSV)
comparacion

,formato,tamaño_MB,lectura_s
0,CSV,730.6,7.7
1,Parquet,67.1,1.2


**Por qué Parquet:** guarda los datos por columna y comprimidos, por eso pesa mucho menos y se lee más rápido.
Además conserva los tipos: al leer el CSV las fechas vuelven como texto; al leer el Parquet vuelven como fecha.

## Arquitectura de datos

### Motor elegido: SQLite

Usamos **SQLite** para el Data Warehouse y lo comparamos con **MongoDB**, la base NoSQL de
documentos vista en clases en la S4. MongoDB es la alternativa más tentadora porque su esquema flexible parece resolver
los cinco cambios de columnas de JHU. Elegimos SQLite por tres razones:

1. **El esquema es estable.** Después de unificar los cinco esquemas, todos los reportes quedan con
   las mismas 16 columnas y un grano fijo: territorio × día. Eso es un modelo en estrella con claves primarias
   y foráneas . El esquema flexible de MongoDB no resuelve los cinco esquemas, solo los esconde:
   cada documento conservaría sus nombres originales (`Long_` y `Longitude`, `Incidence_Rate` e `Incident_Rate`).
2. **Las consultas son analíticas.** El DW debe responder con roll-up, drill-down, slice y dice, es decir,
   uniones entre la tabla de hechos y las dimensiones con `GROUP BY`. Eso es SQL. MongoDB consulta con filtros y
   agregaciones y no está pensado para unir colecciones.
3. **El volumen no justifica NoSQL.** Son 4,3 millones de filas, sin escrituras continuas y con un solo usuario consultando. Lo que justifica NoSQL (volumen masivo, escritura de alta frecuencia, repartir los datos en muchas máquinas) no se da aquí. Además, SQLite viene incluido en
Python y no necesita servidor: permitiendo que el proyecto se reproduzca igual en un computador o en Colab.

| Criterio (S4) | SQLite | MongoDB |
|---|---|---|
| Esquema | Fijo: calza con los datos ya unificados | Flexible: mantendría los cinco esquemas mezclados |
| Consultas | SQL con uniones y `GROUP BY` (OLAP) | Filtros y agregaciones; las uniones son difíciles |
| Operación | Un archivo, sin servidor, viene con Python | Requiere instalar y levantar un servidor |

**antes de elegir NoSQL, hay que descartar que una base relacional resuelva el problema**.
Aquí la resuelve.


## Arquitectura por capas (hoy)

```mermaid
flowchart LR
    subgraph Cruda["Capa cruda (no se guarda)"]
        JHU["JHU · 1.143 reportes diarios<br/>+ tabla UID (CSV raw)"]
        OWID["OWID · vacunación (CSV raw)"]
        BM["Banco Mundial (API JSON)"]
    end
    subgraph Limpia["Capa limpia · data/clean/ (Parquet zstd)"]
        L1["covid_limpio.parquet<br/>4.287.473 filas"]
        L2["covid_diario.parquet<br/>indice_calidad_pais.parquet"]
    end
    subgraph Analitica["Capa analítica · data/dw/"]
        DW[("covid_dw.db · SQLite<br/>estrella: dim_fecha, dim_territorio,<br/>fact_covid, fact_vacunacion")]
    end
    subgraph Consumo["Consumo"]
        NB["Notebooks 04–06<br/>(protección, análisis, minería)"]
        TAB["Tablero Streamlit<br/>dashboard/app.py"]
    end
    JHU -->|01_ingesta| L1
    L1 -->|02_calidad| L2
    L2 -->|03_dw| DW
    OWID -->|03_dw| DW
    BM -->|03_dw| DW
    DW --> NB
    DW --> TAB
```

Es la arquitectura de un Data Warehouse : **fuentes → ETL → almacén → uso**. El ETL son los notebooks 01–03.

## Equivalente en la nube: Google Cloud (GCP), región Santiago

Elegimos **Google Cloud** porque tiene una región en **Santiago de Chile** (`southamerica-west1`): los datos se guardan y procesan
en Chile. Aunque estos datos agregados no son personales , una plataforma de salud real sí podría tratarlos, y tenerlos en el país
simplifica cumplir la Ley 21.719.

```mermaid
flowchart LR
    SRC["Fuentes en internet<br/>JHU · OWID · Banco Mundial"]
    SCH["Cloud Scheduler<br/>(todos los días, 06:00)"]
    JOB["Cloud Run Jobs<br/>ingesta + calidad + carga<br/>(el código de 01–03 en un contenedor)"]
    GCS[("Cloud Storage<br/>capa limpia en Parquet")]
    BQ[("BigQuery<br/>DW en estrella")]
    RUN["Cloud Run<br/>tablero Streamlit"]
    AR["Artifact Registry<br/>imágenes de los contenedores"]
    USR["Comité / usuarios"]
    SCH --> JOB
    SRC --> JOB
    JOB --> GCS
    GCS --> BQ
    BQ --> RUN
    RUN --> USR
    AR -.-> JOB
    AR -.-> RUN
```

| Capa | Hoy | En Google Cloud | Por qué |
|---|---|---|---|
| Cruda | URL raw (no se guarda) | Igual: el job lee desde internet | no duplica las fuentes |
| ETL | Notebooks 01–03, a mano | **Cloud Run Jobs** lanzado por **Cloud Scheduler** cada día | Contenedor que corre y se apaga: se paga solo el tiempo de ejecución |
| Limpia | Parquet en `data/clean/` | **Cloud Storage** (Parquet) | Almacenamiento de archivos barato; BigQuery lee Parquet directo |
| Analítica | SQLite (`covid_dw.db`) | **BigQuery** | DW administrado con SQL; el modelo estrella y las consultas OLAP se mantienen |
| Consumo | Streamlit en el computador | **Cloud Run** (servicio) | El mismo `app.py` en un contenedor con URL pública |
| Imágenes | — | **Artifact Registry** | Guarda los contenedores del job y del tablero |

## Costo mensual estimado

Precios de lista de Google Cloud para `southamerica-west1`, consultados el **21-nov-2026** en las páginas oficiales de precios de
[Cloud Run](https://cloud.google.com/run/pricing), [Cloud Storage](https://cloud.google.com/storage/pricing),
[BigQuery](https://cloud.google.com/bigquery/pricing), [Cloud Scheduler](https://cloud.google.com/scheduler/pricing) y
[Artifact Registry](https://cloud.google.com/artifact-registry/pricing). 

| Precio unitario (Santiago) | USD |
|---|---|
| Cloud Run por instancia y Cloud Run Jobs: CPU / memoria | 0,0000216 por vCPU-segundo / 0,0000024 por GiB-segundo |
| Cloud Run por solicitud (tiempo activo): CPU / memoria | 0,0000336 por vCPU-segundo / 0,0000035 por GiB-segundo |
| Cloud Storage Standard | 0,030 por GiB al mes |
| BigQuery: almacenamiento lógico activo / consultas a pedido | 0,033 por GiB al mes (10 GiB gratis) / 8,94 por TiB (1 TiB gratis al mes) |
| Cloud Scheduler | 0,10 por trabajo al mes (3 gratis por cuenta) |
| Artifact Registry | 0,10 por GiB al mes (0,5 GiB gratis) |

**Supuestos:** capa limpia de ~0,13 GiB; DW de menos de 10 GiB en BigQuery; ingesta diaria completa de 15 minutos con 2 vCPU y 4 GiB
(el notebook 01 usa ~1,3 GB de RAM); tablero con 1 vCPU y 2 GiB; consultas del tablero muy por debajo de 1 TiB al mes; imágenes de
~1,5 GiB; ~2 GiB al mes de tráfico hacia los usuarios (0,12 USD por GiB).

| Componente | A · Tablero siempre encendido | B · Tablero que se apaga sin uso (40 h de uso al mes) |
|---|---|---|
| Tablero (Cloud Run) | 69,38 | 5,85 |
| Ingesta diaria (Cloud Run Jobs: 54.000 vCPU-s y 108.000 GiB-s) | 1,43 | 1,43 |
| Cloud Storage | < 0,01 | < 0,01 |
| BigQuery (almacenamiento y consultas) | 0 (dentro de lo gratuito) | 0 (dentro de lo gratuito) |
| Cloud Scheduler (1 trabajo) | 0 (gratis) | 0 (gratis) |
| Artifact Registry | 0,10 | 0,10 |
| Tráfico de salida | 0,24 | 0,24 |
| **Total a precio de lista** | **≈ 71 USD/mes** | **≈ 8 USD/mes** |
| Total con el nivel gratuito de Cloud Run | ≈ 64 USD/mes | ≈ 1 USD/mes |

**Lectura:** los datos son pocos (menos de 1 GiB), así que almacenar y consultar es casi gratis; **lo que cuesta es tener el tablero
encendido**. Para la sala de crisis conviene el escenario **B** (Cloud Run escala a cero y se enciende al primer acceso, con unos segundos
de espera) y pasar al **A** solo durante una emergencia, cuando el tablero se mira todo el día. 